In [ ]:
# ================================================================
# PHYSICAL AI + LLM + COMPUTER VISION IMAGE DEMO
#
# Demo:
# 1. Upload room image
# 2. Click destination/target
# 3. Gemini creates high-level robot plan
# 4. Robot starts from lower-left
# 5. Robot moves to bottle
# 6. Robot picks up bottle
# 7. Robot carries bottle to selected target
# 8. Robot drops bottle
#
# Google Colab
# ================================================================

!pip -q install pillow google-genai

In [ ]:
import cv2
import base64
import numpy as np

from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont

from IPython.display import display, Image

from google.colab import files
from google.colab.output import eval_js
from google.colab import userdata

from google import genai

In [ ]:
# ================================================================
# STEP 1
# GEMINI SETUP
# ================================================================

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

print("Gemini connected.")

In [ ]:

# ================================================================
# STEP 2
# UPLOAD ROOM IMAGE
# ================================================================

print("\nUpload room.jpg")

uploaded = files.upload()

image_path = next(iter(uploaded))

print(
    "Uploaded:",
    image_path
)


In [ ]:
# ================================================================
# STEP 3
# READ ROOM IMAGE
# ================================================================

room = cv2.imread(
    image_path
)

room = cv2.cvtColor(
    room,
    cv2.COLOR_BGR2RGB
)

height, width, _ = room.shape

print(
    "Room image size:",
    width,
    "x",
    height
)

In [ ]:

# ================================================================
# STEP 4
# USER SELECTS FINAL TARGET
# ================================================================

_, buffer = cv2.imencode(

    ".jpg",

    cv2.cvtColor(
        room,
        cv2.COLOR_RGB2BGR
    )
)

image_base64 = base64.b64encode(
    buffer
).decode()


javascript = f"""
new Promise((resolve) => {{

    const container =
        document.createElement('div');

    container.innerHTML = `
    <h2>
    Select where the robot should deliver the bottle
    </h2>

    <p>
    Click anywhere on the room image.
    </p>

    <canvas
        id="targetCanvas"
        style="
        border:3px solid black;
        cursor:crosshair;">
    </canvas>

    <p id="status">
    Waiting for target selection...
    </p>
    `;

    document.body.appendChild(
        container
    );


    const canvas =
        document.getElementById(
            "targetCanvas"
        );

    const ctx =
        canvas.getContext(
            "2d"
        );


    const img =
        new Image();

    img.src =
        "data:image/jpeg;base64,{image_base64}";


    img.onload = function() {{

        const maxWidth = 900;

        let scale = 1;

        if (
            img.width >
            maxWidth
        ) {{

            scale =
                maxWidth /
                img.width;
        }}


        canvas.width =
            img.width *
            scale;

        canvas.height =
            img.height *
            scale;


        ctx.drawImage(

            img,

            0,

            0,

            canvas.width,

            canvas.height

        );


        canvas.onclick =
        function(event) {{

            const rect =
                canvas.getBoundingClientRect();


            const clickX =
                event.clientX -
                rect.left;

            const clickY =
                event.clientY -
                rect.top;


            const originalX =
                clickX /
                scale;

            const originalY =
                clickY /
                scale;


            ctx.beginPath();

            ctx.arc(

                clickX,

                clickY,

                20,

                0,

                Math.PI * 2

            );


            ctx.strokeStyle =
                "red";

            ctx.lineWidth =
                6;

            ctx.stroke();


            ctx.font =
                "22px Arial";

            ctx.fillStyle =
                "red";


            ctx.fillText(

                "DELIVERY TARGET",

                clickX + 25,

                clickY

            );


            document.getElementById(
                "status"
            ).innerHTML =
                "Target selected";


            resolve([
                originalX,
                originalY
            ]);

        }};

    }};

}})
"""


target_coordinates = eval_js(
    javascript
)


delivery_target = np.array(
    target_coordinates,
    dtype=float
)


print(
    "\nDelivery target selected:",
    delivery_target.astype(int)
)


In [ ]:
# ================================================================
# STEP 5
# DEFINE STARTING POSITION
#
# Robot begins at lower-left
# ================================================================

robot_start = np.array(

    [
        width * 0.10,
        height * 0.82
    ],

    dtype=float

)


# ================================================================
# STEP 6
# BOTTLE LOCATION
#
# For this first demonstration we simulate
# the bottle location.
#
# Later this can be replaced by YOLO detection.
# ================================================================

bottle_position = np.array(

    [
        width * 0.50,
        height * 0.58
    ],

    dtype=float

)


print(
    "Robot start:",
    robot_start.astype(int)
)

print(
    "Bottle location:",
    bottle_position.astype(int)
)


In [ ]:
# ================================================================
# STEP 7
# ASK LLM TO CREATE ROBOT PLAN
# ================================================================

prompt = f"""
You are the high-level planner for a Physical AI
service robot.

Robot starting position:

{robot_start.astype(int).tolist()}

Bottle position:

{bottle_position.astype(int).tolist()}

Delivery target:

{delivery_target.astype(int).tolist()}

The goal is:

1. Move from the starting position to the bottle.
2. Pick up the bottle.
3. Carry the bottle to the selected delivery target.
4. Release the bottle.
5. Stop.

Create a concise numbered robot task plan.

Do not calculate motor control.
Only provide high-level actions.
"""


response = client.models.generate_content(

    model="gemini-2.5-flash",

    contents=prompt

)


print(
    "\n==============================="
)

print(
    "GEMINI ROBOT PLAN"
)

print(
    "===============================\n"
)

print(
    response.text
)

In [ ]:
# ================================================================
# STEP 8
# CREATE PATH:
#
# START -> BOTTLE
# ================================================================

frames_to_bottle = 35


path_to_bottle = []


for t in np.linspace(

    0,

    1,

    frames_to_bottle

):

    position = (

        robot_start *
        (1 - t)

        +

        bottle_position *
        t

    )

    path_to_bottle.append(
        position.copy()
    )


# ================================================================
# STEP 9
# CREATE PATH:
#
# BOTTLE -> DELIVERY TARGET
# ================================================================

frames_to_target = 45


path_to_target = []


for t in np.linspace(

    0,

    1,

    frames_to_target

):

    position = (

        bottle_position *
        (1 - t)

        +

        delivery_target *
        t

    )


    # Slight curve

    position[1] -= (

        40 *

        np.sin(
            np.pi * t
        )

    )


    path_to_target.append(
        position.copy()
    )



In [ ]:

# ================================================================
# STEP 10
# BUILD COMPLETE TRAJECTORY
# ================================================================

trajectory = (

    path_to_bottle

    +

    path_to_target

)


# ================================================================
# STEP 11
# ROBOT DRAWING FUNCTION
# ================================================================

def draw_robot(
    draw,
    x,
    y
):

    robot_width = 58

    robot_height = 64


    # Body

    draw.rounded_rectangle(

        [
            x - robot_width//2,
            y - robot_height//2,

            x + robot_width//2,
            y + robot_height//2
        ],

        radius=12,

        fill="blue",

        outline="white",

        width=4

    )


    # Head

    draw.rounded_rectangle(

        [
            x - 22,
            y - 65,

            x + 22,
            y - 33
        ],

        radius=6,

        fill="lightblue",

        outline="white",

        width=3

    )


    # Eyes

    draw.ellipse(

        [
            x - 13,
            y - 55,

            x - 6,
            y - 48
        ],

        fill="black"

    )


    draw.ellipse(

        [
            x + 6,
            y - 55,

            x + 13,
            y - 48
        ],

        fill="black"

    )


    # Wheels

    draw.ellipse(

        [
            x - 36,
            y + 18,

            x - 20,
            y + 42
        ],

        fill="black"

    )


    draw.ellipse(

        [
            x + 20,
            y + 18,

            x + 36,
            y + 42
        ],

        fill="black"

    )


In [ ]:
# ================================================================
# STEP 12
# GENERATE ANIMATION
# ================================================================

frames = []


holding_bottle = False


for frame_number, position in enumerate(
    trajectory
):


    frame = PILImage.fromarray(
        room.copy()
    )


    draw = ImageDraw.Draw(
        frame
    )


    x = int(
        position[0]
    )

    y = int(
        position[1]
    )


    # ============================================================
    # DRAW FINAL TARGET
    # ============================================================

    tx = int(
        delivery_target[0]
    )

    ty = int(
        delivery_target[1]
    )


    draw.ellipse(

        [
            tx - 28,
            ty - 28,

            tx + 28,
            ty + 28
        ],

        outline="red",

        width=7

    )


    draw.text(

        (
            tx + 30,
            ty
        ),

        "TARGET",

        fill="red"

    )


    # ============================================================
    # DETERMINE WHETHER BOTTLE HAS BEEN PICKED UP
    # ============================================================

    if frame_number >= frames_to_bottle:

        holding_bottle = True


    # ============================================================
    # DRAW BOTTLE BEFORE PICKUP
    # ============================================================

    if not holding_bottle:

        bx = int(
            bottle_position[0]
        )

        by = int(
            bottle_position[1]
        )


        draw.rectangle(

            [
                bx - 10,
                by - 30,

                bx + 10,
                by + 20
            ],

            fill="green",

            outline="white",

            width=3

        )


        draw.text(

            (
                bx + 15,
                by - 10
            ),

            "BOTTLE",

            fill="green"

        )


    # ============================================================
    # DRAW ROBOT
    # ============================================================

    draw_robot(
        draw,
        x,
        y
    )


    # ============================================================
    # DRAW BOTTLE IN ROBOT'S HAND
    # ============================================================

    if holding_bottle:

        draw.rectangle(

            [
                x + 30,
                y - 20,

                x + 42,
                y + 15
            ],

            fill="green",

            outline="white",

            width=2

        )


    # ============================================================
    # DRAW PATH
    # ============================================================

    if frame_number > 1:

        previous_positions = (

            trajectory[
                :frame_number + 1
            ]

        )


        path_points = [

            (
                int(p[0]),
                int(p[1])
            )

            for p in previous_positions

        ]


        draw.line(

            path_points,

            fill="yellow",

            width=5

        )


        # Redraw robot over path

        draw_robot(
            draw,
            x,
            y
        )


        if holding_bottle:

            draw.rectangle(

                [
                    x + 30,
                    y - 20,

                    x + 42,
                    y + 15
                ],

                fill="green",

                outline="white",

                width=2

            )


    # ============================================================
    # ACTION STATUS
    # ============================================================

    if frame_number < frames_to_bottle - 3:

        status = (

            "ACTION: Moving toward bottle"

        )


    elif frame_number < frames_to_bottle:

        status = (

            "ACTION: Picking up bottle"

        )


    elif frame_number < len(
        trajectory
    ) - 3:

        status = (

            "ACTION: Carrying bottle to target"

        )


    else:

        status = (

            "ACTION: Releasing bottle"

        )


    draw.text(

        (
            20,
            20
        ),

        status,

        fill="white",

        stroke_width=3,

        stroke_fill="black"

    )


    draw.text(

        (
            20,
            50
        ),

        f"Step {frame_number}",

        fill="white",

        stroke_width=2,

        stroke_fill="black"

    )


    frames.append(
        frame
    )

In [ ]:

# ================================================================
# STEP 13
# FINAL FRAMES:
#
# DROP BOTTLE AT TARGET
# ================================================================

final_frame = frames[-1].copy()

draw = ImageDraw.Draw(
    final_frame
)


tx = int(
    delivery_target[0]
)

ty = int(
    delivery_target[1]
)


# Bottle placed at target

draw.rectangle(

    [
        tx - 10,
        ty - 25,

        tx + 10,
        ty + 20
    ],

    fill="green",

    outline="white",

    width=3

)


draw.text(

    (
        20,
        20
    ),

    "TASK COMPLETE: Bottle delivered!",

    fill="yellow",

    stroke_width=3,

    stroke_fill="black"

)


# Hold final result for several frames

for _ in range(12):

    frames.append(
        final_frame.copy()
    )


# ================================================================
# STEP 14
# SAVE GIF
# ================================================================

output_file = (

    "/content/"
    "physical_ai_llm_robot_demo.gif"

)


frames[0].save(

    output_file,

    save_all=True,

    append_images=frames[1:],

    duration=140,

    loop=0

)


print(
    "\nAnimation created."
)


# ================================================================
# STEP 15
# DISPLAY ANIMATION
# ================================================================

display(

    Image(
        filename=output_file
    )

)